In [9]:
# Load the fixed sampled dataset from disk
import json
from datasets import load_dataset
from pathlib import Path

data_path = Path("/home/a/arfaoui/rag_project/data/hotpotqa_sample_500.json")

dataset = load_dataset("json", data_files=str(data_path))["train"]

print("Loaded sample size:", len(dataset))
print("Columns:", dataset.column_names)



Loaded sample size: 500
Columns: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']


In [18]:
# We build a corpus from the HotpotQA contexts.
# Each document will be:
# - title + joined sentences
# - converted into a single text string

corpus = []
seen_titles = set()  # will be used to avoid duplicate articles
duplicate_count = 0
for example in dataset:
    context = example["context"]

    # IMPORTANT:
    # context is a dict with two parallel lists:
    # - context["title"]
    # - context["sentences"]
    # Observations from previous NoteBook

    for title, sentences in zip(context["title"], context["sentences"]):
         if title not in seen_titles:
            seen_titles.add(title)
            text = " ".join(sentences) # Join all sentences into one text block
            corpus.append({            # Store structured document
               "title": title,
               "text": text
        })
         else:
            duplicate_count += 1

# Print to check
print("Total documents in corpus no duplicates:", len(corpus))
print("\nExample document:\n")
print(corpus[0])
print("Number of duplicates:", duplicate_count)

Total documents in corpus: 4962

Example document:

{'title': 'Wedding Dress (film)', 'text': 'Wedding Dress is a South Korean drama film, released on January 14, 2010.'}
Number of duplicates: 27


In [22]:
# Save the corpus in data folder 
import json
from pathlib import Path

corpus_path = Path("/home/a/arfaoui/rag_project/data/corpus.json")

with open(corpus_path, "w", encoding="utf-8") as f:   # ensure_ascii=False -> important for any non-ASCII characters in HotpotQA
    json.dump(corpus, f, ensure_ascii=False, indent=2)

print("Corpus saved to:", corpus_path)

Corpus saved to: /home/a/arfaoui/rag_project/data/corpus.json


### Corpus Construction Observations

- Each HotpotQA example contains multiple documents (≈10).
- Each document is defined by:
  - a title
  - a list of sentences
- We reconstruct documents by joining sentences into one text.
- Removing duplicates keeps the retrieval index cleaner and avoids storing the same article multiple times.
  
Result:
- The sampled 500 HotpotQA questions contained 4,989 document occurrences in total.
- After deduplication by title, the final corpus contains 4,962 unique documents.
- 27 duplicate article occurrences were removed.

  
Why this matters:
- This corpus is the basis for embedding + retrieval.
# Errors in this step break the entire RAG pipeline.